In [ ]:
# ══════════════════════════════════════════════════════════
# SECTION 05 - STATISTICAL ANALYSIS
# Bangkok Airbnb Market Intelligence
# ══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

# ── Plot styling ───────────────────────────────────────────
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size']      = 12
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("husl")

PLOTS_DIR  = '../reports/plots/'
CLEAN_PATH = '../data/bangkok/cleaned/'

# ── Load data ──────────────────────────────────────────────
master = pd.read_csv(CLEAN_PATH + 'listings_master.csv')
master['host_since']   = pd.to_datetime(master['host_since'],   errors='coerce')
master['first_review'] = pd.to_datetime(master['first_review'], errors='coerce')
master['last_review']  = pd.to_datetime(master['last_review'],  errors='coerce')

print("Data loaded!")
print(f"master: {master.shape}")
print(f"\nColumns available: {list(master.columns[:20])}")

In [ ]:
print(f"Memory usage: {master.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Keep only columns we need for statistical analysis
cols_needed = [
    'id', 'host_id', 'room_type', 'property_category',
    'neighbourhood_standard', 'price_numeric', 'price_flag',
    'occupancy_rate', 'availability_365', 'minimum_nights',
    'number_of_reviews', 'review_scores_rating',
    'review_scores_cleanliness', 'review_scores_location',
    'review_scores_communication', 'review_scores_value',
    'host_is_superhost', 'host_tenure_years', 'bedrooms',
    'accommodates', 'estimated_annual_revenue',
    'is_commercial_host', 'total_reviews'
]

# Keep only needed columns
master = master[cols_needed].copy()

print(f"After column selection: {master.shape}")
print(f"Memory usage: {master.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size']      = 12
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

PLOTS_DIR = '../reports/plots/'

print("Libraries loaded!")

# ── Helper function for hypothesis tests ───────────────────
def report_test(hypothesis, null, alternative, test_name,
                statistic, p_value, effect_size, effect_label,
                conclusion, business_meaning):
    print(f"\n{'='*60}")
    print(f"HYPOTHESIS: {hypothesis}")
    print(f"{'='*60}")
    print(f"  H0 (Null):        {null}")
    print(f"  H1 (Alternative): {alternative}")
    print(f"  Test:             {test_name}")
    print(f"  Statistic:        {statistic:.4f}")
    print(f"  P-value:          {p_value:.6f}")
    print(f"  Effect Size:      {effect_size:.4f} ({effect_label})")
    print(f"  Alpha:            0.05")
    print(f"  Decision:         {'REJECT H0' if p_value < 0.05 else 'FAIL TO REJECT H0'}")
    print(f"  Conclusion:       {conclusion}")
    print(f"  Business Meaning: {business_meaning}")

print("Helper function ready!")

In [ ]:
# ══════════════════════════════════════════════════════════
# 5.1 HYPOTHESIS TESTING
# ══════════════════════════════════════════════════════════

# ── H1: Entire-home listings command higher prices ─────────
entire = master[
    (master['room_type'] == 'Entire home/apt') &
    (master['price_numeric'] > 0) &
    (master['price_numeric'] <= 10000)
]['price_numeric']

private = master[
    (master['room_type'] == 'Private room') &
    (master['price_numeric'] > 0) &
    (master['price_numeric'] <= 10000)
]['price_numeric']

# Mann-Whitney U test (non-parametric, prices not normally distributed)
stat, p = stats.mannwhitneyu(entire, private, alternative='greater')

# Effect size: rank-biserial correlation
n1, n2 = len(entire), len(private)
effect = 1 - (2 * stat) / (n1 * n2)

report_test(
    hypothesis    = "H1: Entire-home listings command higher prices than private rooms",
    null          = "Entire-home prices <= Private room prices",
    alternative   = "Entire-home prices > Private room prices",
    test_name     = "Mann-Whitney U (non-parametric, one-tailed)",
    statistic     = stat,
    p_value       = p,
    effect_size   = effect,
    effect_label  = "Rank-biserial correlation",
    conclusion    = f"Entire home median=฿{entire.median():,.0f}, Private room median=฿{private.median():,.0f}",
    business_meaning = "Entire home listings price premium is statistically confirmed. "
                      "Hosts with entire properties should position as premium offerings."
)

# ── H2: Superhost listings achieve higher review scores ────
super_ratings = master[
    (master['host_is_superhost'] == 't') &
    (master['review_scores_rating'] > 0)
]['review_scores_rating']

regular_ratings = master[
    (master['host_is_superhost'] == 'f') &
    (master['review_scores_rating'] > 0)
]['review_scores_rating']

stat2, p2 = stats.mannwhitneyu(super_ratings, regular_ratings, alternative='greater')
effect2 = 1 - (2 * stat2) / (len(super_ratings) * len(regular_ratings))

report_test(
    hypothesis    = "H2: Superhost listings achieve higher review scores",
    null          = "Superhost ratings <= Regular host ratings",
    alternative   = "Superhost ratings > Regular host ratings",
    test_name     = "Mann-Whitney U (non-parametric, one-tailed)",
    statistic     = stat2,
    p_value       = p2,
    effect_size   = effect2,
    effect_label  = "Rank-biserial correlation",
    conclusion    = f"Superhost median={super_ratings.median():.2f}, Regular median={regular_ratings.median():.2f}",
    business_meaning = "Superhost status is a reliable signal of quality. "
                      "Guests can trust Superhost badge as quality indicator."
)

# ── H3: Listings with >10 reviews have different prices ────
many_reviews = master[
    (master['number_of_reviews'] > 10) &
    (master['price_numeric'] > 0) &
    (master['price_numeric'] <= 10000)
]['price_numeric']

few_reviews = master[
    (master['number_of_reviews'] <= 10) &
    (master['price_numeric'] > 0) &
    (master['price_numeric'] <= 10000)
]['price_numeric']

stat3, p3 = stats.mannwhitneyu(many_reviews, few_reviews, alternative='two-sided')
effect3 = 1 - (2 * stat3) / (len(many_reviews) * len(few_reviews))

report_test(
    hypothesis    = "H3: Listings with >10 reviews have different prices",
    null          = "Prices are equal regardless of review count",
    alternative   = "Prices differ based on review count (two-tailed)",
    test_name     = "Mann-Whitney U (non-parametric, two-tailed)",
    statistic     = stat3,
    p_value       = p3,
    effect_size   = abs(effect3),
    effect_label  = "Rank-biserial correlation",
    conclusion    = f">10 reviews median=฿{many_reviews.median():,.0f}, <=10 reviews median=฿{few_reviews.median():,.0f}",
    business_meaning = "Popular listings (>10 reviews) tend to be priced differently. "
                      "Review volume is correlated with price positioning."
)

# ── H4: Neighbourhood prices differ significantly ──────────
# ANOVA across top 10 neighbourhoods
top10_nb = master['neighbourhood_standard'].value_counts().head(10).index
nb_groups = [
    master[
        (master['neighbourhood_standard'] == nb) &
        (master['price_numeric'] > 0) &
        (master['price_numeric'] <= 10000)
    ]['price_numeric'].values
    for nb in top10_nb
]

stat4, p4 = stats.kruskal(*nb_groups)
# Eta-squared approximation for Kruskal-Wallis
n_total = sum(len(g) for g in nb_groups)
effect4 = (stat4 - len(nb_groups) + 1) / (n_total - len(nb_groups))

report_test(
    hypothesis    = "H4: Neighbourhood average prices differ significantly",
    null          = "All neighbourhood median prices are equal",
    alternative   = "At least one neighbourhood has different median price",
    test_name     = "Kruskal-Wallis H test (non-parametric ANOVA)",
    statistic     = stat4,
    p_value       = p4,
    effect_size   = effect4,
    effect_label  = "Eta-squared approximation",
    conclusion    = "Significant price variation exists across Bangkok neighbourhoods",
    business_meaning = "Location premium is statistically real. Neighbourhood choice "
                      "significantly impacts achievable rental price."
)

# ── H5: Weekend vs weekday pricing ────────────────────────

import duckdb

con = duckdb.connect('../data/bangkok/airbnb_bangkok.duckdb')

weekend_agg = con.execute("""
    SELECT
        CASE WHEN DAYOFWEEK(CAST(date AS DATE)) IN (1,7) 
             THEN 'weekend' ELSE 'weekday' END AS day_type,
        COUNT(*)                               AS total_days,
        SUM(CASE WHEN available = 'f' 
            THEN 1 ELSE 0 END)                AS booked_days
    FROM read_csv_auto('../data/bangkok/cleaned/calendar_clean.csv')
    WHERE date IS NOT NULL
    GROUP BY day_type
""").df()
con.close()

weekend_agg['occupancy_pct'] = (
    weekend_agg['booked_days'] / weekend_agg['total_days'] * 100
).round(2)

print("Weekend vs Weekday Booking Rates:")
print(weekend_agg)

# Chi-square test on booking counts
weekend_row = weekend_agg[weekend_agg['day_type'] == 'weekend'].iloc[0]
weekday_row = weekend_agg[weekend_agg['day_type'] == 'weekday'].iloc[0]

observed = np.array([
    [weekend_row['booked_days'],  weekend_row['total_days']  - weekend_row['booked_days']],
    [weekday_row['booked_days'],  weekday_row['total_days']  - weekday_row['booked_days']]
])

stat5, p5, dof, expected = stats.chi2_contingency(observed)
effect5 = np.sqrt(stat5 / observed.sum())

report_test(
    hypothesis    = "H5: Weekend vs weekday booking rates differ",
    null          = "Weekend and weekday booking rates are equal",
    alternative   = "Weekend and weekday booking rates differ",
    test_name     = "Chi-Square test of independence",
    statistic     = stat5,
    p_value       = p5,
    effect_size   = effect5,
    effect_label  = "Cramer's V",
    conclusion    = f"Weekend occupancy={weekend_row['occupancy_pct']:.1f}%, "
                   f"Weekday occupancy={weekday_row['occupancy_pct']:.1f}%",
    business_meaning = "Weekend vs weekday demand difference guides dynamic pricing strategy."
)

In [ ]:
# ══════════════════════════════════════════════════════════
# 5.2 CONFIDENCE INTERVALS & EFFECT SIZES
# ══════════════════════════════════════════════════════════
print("=" * 60)
print("CONFIDENCE INTERVALS - Mean Price by Room Type")
print("=" * 60)

def confidence_interval(data, confidence=0.95):
    n    = len(data)
    mean = np.mean(data)
    se   = stats.sem(data)
    ci   = stats.t.interval(confidence, df=n-1, loc=mean, scale=se)
    return mean, ci[0], ci[1], n

room_types = ['Entire home/apt', 'Private room', 'Hotel room', 'Shared room']

print(f"\n{'Room Type':<20} {'N':>6} {'Mean':>10} {'95% CI Lower':>14} {'95% CI Upper':>14}")
print("-" * 70)

ci_results = []
for room in room_types:
    data = master[
        (master['room_type'] == room) &
        (master['price_numeric'] > 0) &
        (master['price_numeric'] <= 10000)
    ]['price_numeric'].dropna()

    if len(data) > 1:
        mean, lower, upper, n = confidence_interval(data)
        ci_results.append({
            'room_type': room, 'n': n,
            'mean': mean, 'lower': lower, 'upper': upper
        })
        print(f"{room:<20} {n:>6,} {mean:>10,.0f} {lower:>14,.0f} {upper:>14,.0f}")

print("\n" + "=" * 60)
print("CONFIDENCE INTERVALS - Mean Price by Top 10 Neighbourhoods")
print("=" * 60)

top10 = master['neighbourhood_standard'].value_counts().head(10).index
print(f"\n{'Neighbourhood':<20} {'N':>6} {'Mean':>10} {'95% CI Lower':>14} {'95% CI Upper':>14}")
print("-" * 70)

nb_ci_results = []
for nb in top10:
    data = master[
        (master['neighbourhood_standard'] == nb) &
        (master['price_numeric'] > 0) &
        (master['price_numeric'] <= 10000)
    ]['price_numeric'].dropna()

    if len(data) > 1:
        mean, lower, upper, n = confidence_interval(data)
        nb_ci_results.append({
            'neighbourhood': nb, 'n': n,
            'mean': mean, 'lower': lower, 'upper': upper
        })
        print(f"{nb:<20} {n:>6,} {mean:>10,.0f} {lower:>14,.0f} {upper:>14,.0f}")

# Cohen's d between entire home and private room
entire = master[
    (master['room_type'] == 'Entire home/apt') &
    (master['price_numeric'] > 0) &
    (master['price_numeric'] <= 10000)
]['price_numeric'].dropna()

private = master[
    (master['room_type'] == 'Private room') &
    (master['price_numeric'] > 0) &
    (master['price_numeric'] <= 10000)
]['price_numeric'].dropna()

pooled_std = np.sqrt(
    ((len(entire)-1)*entire.std()**2 + (len(private)-1)*private.std()**2) /
    (len(entire) + len(private) - 2)
)
cohens_d = (entire.mean() - private.mean()) / pooled_std

print(f"\n{'='*60}")
print(f"EFFECT SIZE: Cohen's d (Entire home vs Private room)")
print(f"{'='*60}")
print(f"  Cohen's d = {cohens_d:.4f}")
print(f"  Interpretation: {'Small' if abs(cohens_d) < 0.5 else 'Medium' if abs(cohens_d) < 0.8 else 'Large'} effect")
print(f"\n  Note: Statistical significance (p<0.05) confirmed in H1,")
print(f"  but Cohen's d={cohens_d:.2f} indicates a small-to-medium practical effect.")
print(f"  The price difference (฿{entire.mean()-private.mean():,.0f}) is real but")
print(f"  context-dependent — luxury private rooms may exceed budget entire homes.")

In [ ]:
# ══════════════════════════════════════════════════════════
# 5.3 CORRELATION & DRIVER ANALYSIS
# ══════════════════════════════════════════════════════════
print("=" * 60)
print("CORRELATION MATRIX - Numerical Features")
print("=" * 60)

# Select numerical columns for correlation
corr_cols = [
    'price_numeric', 'occupancy_rate', 'availability_365',
    'minimum_nights', 'number_of_reviews', 'review_scores_rating',
    'review_scores_cleanliness', 'review_scores_location',
    'review_scores_value', 'host_tenure_years',
    'bedrooms', 'accommodates', 'total_reviews'
]

# Filter valid data
corr_data = master[corr_cols].copy()
corr_data = corr_data.replace(-1, np.nan)
corr_data = corr_data[corr_data['price_numeric'] <= 10000]
corr_data = corr_data[corr_data['price_numeric'] > 0]

corr_matrix = corr_data.corr()

# Plot correlation heatmap
fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Figure 6: Correlation Matrix - Key Numerical Features',
             fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../reports/plots/fig06_correlation_matrix.png', 
            dpi=150, bbox_inches='tight')
plt.show()

# Print strongest correlations with price
print("\nStrongest correlations with PRICE:")
price_corr = corr_matrix['price_numeric'].drop('price_numeric').sort_values(
    key=abs, ascending=False)
print(price_corr.to_string())

In [ ]:
# ── OLS REGRESSION ─────────────────────────────────────────
print("=" * 60)
print("OLS REGRESSION - Price Drivers")
print("=" * 60)

# Prepare regression data
reg_data = master[
    (master['price_numeric'] > 0) &
    (master['price_numeric'] <= 10000)
].copy()

# Encode categorical variables
reg_data['is_entire_home'] = (reg_data['room_type'] == 'Entire home/apt').astype(int)
reg_data['is_private_room'] = (reg_data['room_type'] == 'Private room').astype(int)
reg_data['is_superhost'] = (reg_data['host_is_superhost'] == 't').astype(int)
reg_data['review_scores_rating'] = reg_data['review_scores_rating'].replace(-1, np.nan)

# Select features
features = [
    'accommodates', 'bedrooms', 'is_entire_home',
    'is_private_room', 'is_superhost', 'host_tenure_years',
    'minimum_nights', 'review_scores_rating'
]

reg_clean = reg_data[features + ['price_numeric']].dropna()

X = reg_clean[features]
y = reg_clean['price_numeric']

# Add constant
X_const = sm.add_constant(X)

# Fit OLS
model = sm.OLS(y, X_const).fit()
print(model.summary())

# VIF check
print("\n" + "=" * 60)
print("MULTICOLLINEARITY CHECK - VIF Scores")
print("=" * 60)
vif_data = pd.DataFrame()
vif_data['Feature'] = features
vif_data['VIF'] = [
    variance_inflation_factor(X.values, i) 
    for i in range(X.shape[1])
]
vif_data['Concern'] = vif_data['VIF'].apply(
    lambda x: 'HIGH' if x > 10 else 'MODERATE' if x > 5 else 'OK'
)
print(vif_data.to_string())

In [ ]:
# ── Fix multicollinearity & visualize results ──────────────
# Remove high VIF features
features_fixed = [
    'accommodates', 'bedrooms', 'is_entire_home',
    'is_private_room', 'is_superhost',
    'host_tenure_years', 'minimum_nights'
]

X2       = reg_clean[features_fixed]
X2_const = sm.add_constant(X2)
model2   = sm.OLS(y[:len(X2)], X2_const).fit()

print("Fixed Model R-squared:", round(model2.rsquared, 4))
print("Fixed Model Adj R-squared:", round(model2.rsquared_adj, 4))

# ── Plot regression coefficients ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Coefficient plot
coefs = model2.params.drop('const')
cis   = model2.conf_int().drop('const')
colors = ['#d32f2f' if v < 0 else '#1976d2' for v in coefs]

axes[0].barh(coefs.index, coefs.values, color=colors, alpha=0.85)
axes[0].errorbar(coefs.values, range(len(coefs)),
                 xerr=[coefs.values - cis[0].values,
                       cis[1].values - coefs.values],
                 fmt='none', color='black', capsize=4)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Figure 7a: OLS Regression Coefficients\n(effect on price per night)',
                  fontweight='bold')
axes[0].set_xlabel('Coefficient Value (THB)')

# Residual plot
y_pred = model2.fittedvalues
residuals = model2.resid
axes[1].scatter(y_pred, residuals, alpha=0.3, s=10, color='steelblue')
axes[1].axhline(0, color='red', linewidth=1.5, linestyle='--')
axes[1].set_title('Figure 7b: Residual Plot', fontweight='bold')
axes[1].set_xlabel('Fitted Values (THB)')
axes[1].set_ylabel('Residuals')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'฿{x:,.0f}'))
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'฿{x:,.0f}'))

plt.tight_layout()
plt.savefig('../reports/plots/fig07_regression.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
BUSINESS INTERPRETATION - Regression:
Property size (accommodates + bedrooms) are the strongest price drivers,
together explaining most of the 40% variance captured. Room type adds
significant premium — entire homes command ฿729 more than shared rooms.
Superhost status is NOT statistically significant after controlling for
other factors, suggesting guests value property characteristics over host
badge. The residual plot shows heteroscedasticity — errors increase with
price — meaning our model performs better for budget listings than luxury.

BUSINESS ACTION: Hosts wanting to maximize revenue should focus on 
property size and room type positioning rather than chasing Superhost
status alone. Larger properties with more bedrooms offer the clearest
path to premium pricing.
""")